In [ ]:
import sys
import os
from pathlib import Path

ROOT = Path().resolve().parent.parent
sys.path.insert(0, str(ROOT))

print("Añadido al path:", ROOT)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.exceptions as px
import folium
from scipy import stats

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10,6)

pd.set_option("display.max_columns", None)

In [ ]:
import io

from utils.funciones_minio import crear_cliente_minio, bajar_minio,bajar_mapa_minio
from utils.config import PATH_PRIMARIOS_LIMPIO

OBJ_VIVIENDAS_VENTA = "viviendas_venta"
OBJ_VIVIENDAS_ALQUILER = "viviendas_alquiler"

In [ ]:
client = crear_cliente_minio()

In [ ]:
df_venta = bajar_mapa_minio(client, "rejillas", OBJ_VIVIENDAS_VENTA)
if not isinstance(df_venta, pd.DataFrame):
        df_venta = pd.read_parquet(io.BytesIO(df_venta))
df_alquiler = bajar_mapa_minio(client, "rejillas", OBJ_VIVIENDAS_ALQUILER)
if not isinstance(df_alquiler, pd.DataFrame):
        df_alquiler = pd.read_parquet(io.BytesIO(df_alquiler))
gdf_barrios = bajar_mapa_minio(client,"rejillas","barrios")
gdf_secciones = bajar_mapa_minio(client,"rejillas","secciones censales")
gdf_hexagonos_g = bajar_mapa_minio(client,"rejillas","hexagonos_1")
gdf_hexagonos_p = bajar_mapa_minio(client,"rejillas","hexagonos_2")

## 1. Resumen del dataset
Descripción general de cada dataset: tipos, nulos y valores únicos

In [ ]:
datasets = {
    "Venta":    df_venta,
    "Alquiler": df_alquiler,
}

for nombre, df in datasets.items():
    print(f"RESUMEN — {nombre}")

    resumen = pd.DataFrame({
        "tipo":      df.dtypes,
        "nulos_pct": (df.isnull().mean() * 100).round(2),
        "n_unicos":  df.nunique(),
        "ejemplo":   df.iloc[0],
    }).sort_values("nulos_pct", ascending=False)
    display(resumen)

## 2. Distribución del precio y outliers
Se visualiza la distribución original, los outliers mediante boxplot,
y la distribución tras aplicar el filtro por percentiles 1–99.
Se usa este criterio en lugar de z-score porque la distribución del precio
inmobiliario es fuertemente asimétrica y no sigue una normal.

In [ ]:
for nombre, df in datasets.items():
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    print(df["Precio"].min())
    print(df["Precio"].max())
    axes[0].hist(df["Precio"], bins=100, color="blue", edgecolor="none")
    axes[0].set_title(f"[{nombre}] Precio original")

    axes[1].boxplot(df["Precio"])
    axes[1].set_title(f"[{nombre}] Boxplot")

    q1, q99 = df["Precio"].quantile([0.01, 0.99])
    df_filtrado = df[(df["Precio"] >= q1) & (df["Precio"] <= q99)]
    axes[2].hist(df_filtrado["Precio"], bins=100, color="green", edgecolor="none")
    axes[2].set_title(f"[{nombre}] Filtrado (p1={q1:,.0f} – p99={q99:,.0f} €)")

    plt.suptitle(f"Distribución del Precio — {nombre}", y=1.02)
    plt.tight_layout()
    plt.show()

## 3. Limpieza de los datasets
En un primer paso quitamos individuos que si les falta esa columna sería inútiles. También quitamos pisos repetidos.
En un segundo paso filtramos solo los outliers del percentil 99, ya que es la zona donde se ubican los outliers. 
En un tercer paso quitamos columnas que no aportan nada.
Y en un cuarto paso quitamos columnas con muchos nulos o con valores iguales para todos los datos.

In [ ]:
def limpiar_dataset(df, nombre_mercado):
    print(f"\n[{nombre_mercado.upper()}] Iniciando limpieza...")
    df_limpio = df.copy()
    print(f"  -> Dimensiones iniciales: {df_limpio.shape}")

    # 1. Nulos críticos y duplicados
    filas_inicio = len(df_limpio)
    df_limpio = df_limpio.dropna(subset=['Precio', 'lat', 'lon', 'Superficie'])
    filas_sin_nulos = len(df_limpio)
    nulos_eliminados = filas_inicio - filas_sin_nulos
    
    df_limpio = df_limpio.drop_duplicates(subset=['lat', 'lon', 'Precio', 'Superficie'], keep='first')
    duplicados_eliminados = filas_sin_nulos - len(df_limpio)
    
    print(f"  -> Paso 1: Eliminadas {nulos_eliminados} filas por nulos críticos.")
    print(f"  -> Paso 1: Eliminadas {duplicados_eliminados} filas por ser duplicadas.")

    # 2. Filtrado de outliers (percentil 99)
    def filtrar_outliers(data, col):
        q_high = data[col].quantile(0.99)
        return data[col] <= q_high

    filas_antes_outliers = len(df_limpio)
    mascara = filtrar_outliers(df_limpio, 'Precio')
    df_limpio = df_limpio[mascara]
    outliers_eliminados = filas_antes_outliers - len(df_limpio)
    
    print(f"  -> Paso 2: Eliminadas {outliers_eliminados} filas por ser outliers en 'Precio' (> P99).")

    # 3. Fuga de datos e identificadores
    cols_fuga = ['id', 'Nombre', 'Direccion', 'Tipo_OSM', 'geometry', 'Precio_m2', 
                 'Media_precio_venta', 'Media_precio_m2_venta', 
                 'Media_precio_alquiler', 'Media_precio_m2_alquiler']
    
    cols_a_borrar = [c for c in cols_fuga if c in df_limpio.columns]
    df_limpio = df_limpio.drop(columns=cols_a_borrar)
    print(f"  -> Paso 3: Eliminadas {len(cols_a_borrar)} columnas por riesgo de fuga o identificadores.")
    if cols_a_borrar:
        print(f"     Columnas: {cols_a_borrar}")

    # 4. Columnas con demasiados nulos (>40%) o varianza cero (>99% mismo valor)
    cols_nulos = df_limpio.columns[df_limpio.isnull().mean() > 0.40].tolist()
    cols_zero_var = [col for col in df_limpio.columns if df_limpio[col].value_counts(normalize=True).iloc[0] > 0.99]
    
    df_limpio = df_limpio.drop(columns=cols_nulos + cols_zero_var)
    
    print(f"  -> Paso 4: Eliminadas {len(cols_nulos)} columnas por exceso de nulos (>40%).")
    if cols_nulos:
        print(f"     Columnas: {cols_nulos}")
        
    print(f"  -> Paso 4: Eliminadas {len(cols_zero_var)} columnas por varianza cero (>99%).")
    if cols_zero_var:
        print(f"     Columnas: {cols_zero_var}")

    print(f"  -> Dimensiones finales: {df_limpio.shape}")
    return df_limpio

df_venta_limpio = limpiar_dataset(df_venta, "Venta")
df_alquiler_limpio = limpiar_dataset(df_alquiler, "Alquiler")

datasets_limpios = {
    "Venta":    (df_venta,    df_venta_limpio),
    "Alquiler": (df_alquiler, df_alquiler_limpio),
}

## 4. Análisis de categoricos
Usamos el test kruskal en vez de ANOVA, ya que ANOVA requiere el supuesto de normalidad y nuestros datos no lo cumplen.

In [ ]:
def analizar_y_limpiar_categoricas_kruskal(df, nombre_mercado):
    print(f"TEST KRUSKAL: MERCADO DE {nombre_mercado.upper()}")
    
    # Trabajamos sobre una copia para aislar el proceso
    df_temp_process = df.copy()

    # Identificación de variables no numéricas
    cols_cat = df_temp_process.select_dtypes(include=['object', 'str', 'bool']).columns.tolist()

    # 2. Análisis estadístico de varianza (ANOVA)
    print(f"{'COLUMNA':<20} | {'ÚNICOS':<8} | {'P-VALOR':<10} | {'ESTADO'}")
    print("-" * 60)
    
    for col in cols_cat:
        unicos = df_temp_process[col].nunique()
        df_test = df_temp_process.dropna(subset=[col, 'Precio'])
        
        # Agrupación de la variable objetivo según categorías
        grupos = [grupo['Precio'].values for nombre, grupo in df_test.groupby(col)]
        
        if len(grupos) > 1:
            f_stat, p_valor = stats.kruskal(*grupos)
            
            estado = "OK"
            if p_valor >= 0.05:
                estado = "DESCARTAR (p-valor alto)"
            
            if unicos > 100:
                if estado == "OK":
                    estado = "ELIMINAR (Alta cardinalidad)"
                else:
                    estado += " + ALTA CARDINALIDAD"
                
            print(f"{col:<20} | {unicos:<8} | {p_valor:<10.5f} | {estado}")

    # 3. Eliminación de variables con alta cardinalidad y ruido
    cols_alta_card = ['Calle', 'Tipo_Via', 'Barrio', 'dist_al_edificio']
    df_final = df_temp_process.drop(columns=[c for c in cols_alta_card if c in df_temp_process.columns])

    # 4. Resumen de variables para One-Hot Encoding
    cat_definitivas = df_final.select_dtypes(include=['object', 'str', 'bool']).columns.tolist()
    print("\nVariables categóricas retenidas para el modelo:")
    print(cat_definitivas)
    
    return df_final

df_venta_limpio = analizar_y_limpiar_categoricas_kruskal(df_venta_limpio, "Venta")
df_alquiler_limpio = analizar_y_limpiar_categoricas_kruskal(df_alquiler_limpio, "Alquiler")

In [ ]:
# ¿Cuántos pisos sin cocina/equipamiento hay por rango de superficie?
df_alquiler_limpio["rango_sup"]= pd.cut(df_alquiler_limpio["Superficie"], bins=[0, 30, 60, 100, 200, 9999],
                          labels=["<30", "30-60", "60-100", "100-200", ">200"])

print(df_alquiler_limpio.groupby("rango_sup")["Cocina"].value_counts(normalize=True).unstack())
print(df_alquiler_limpio.groupby("rango_sup")["Equipamiento"].value_counts(normalize=True).unstack())

df_alquiler_limpio = df_alquiler_limpio.drop(columns=["rango_sup"])

Visto esto, eliminaremos la variable Cocina por su escasa varianza, independientemente del p-valor. Pero para el equipamiento aunque tenga un pvalor alto, Kruskal-Wallis es un test univariante que no detecta los efectos mediados por una tercera variable, que en este caso sería la superficie. Ya que el 88% de los pisos pequeños (<30 m²) se alquilan equipados, frente a solo el 34% de los pisos grandes (>200 m²). Esta relación inversa entre equipamiento y superficie hace que los efectos se cancelen cuando el test analiza equipamiento de forma aislada. Por lo que de momento la conservamos.

In [ ]:
df_alquiler_limpio = df_alquiler_limpio.drop(columns=["Cocina"])

## 5. Análisis de correlaciones
Se usa Pearson, para relaciones lineales, Spearman no asume normalidad ni linealidad, mide si existe una relación monótona entre las variables, es decir, si cuando una sube la otra tiende a subir o bajar de forma consistente, aunque no sea proporcional. Y Kendall opera de forma similar a Spearman pero es más robusto cuando hay empates en los datos o cuando el tamaño muestral es moderado.

In [ ]:
def analizar_correlaciones_e_importancia(df, nombre_mercado):
    print(f"ANÁLISIS PREDICTIVO DE VARIABLES: MERCADO DE {nombre_mercado.upper()}")

    # Filtramos solo numéricas y quitamos nulos para que los algoritmos matemáticos no fallen
    df_num = df.select_dtypes(include=['int64', 'float64']).dropna()

    # 1. Pearson (Lineal)
    print("\n--- PEARSON (Relaciones Lineales) ---")
    corr_pearson = df_num.corr()['Precio'].sort_values(ascending=False)
    print("Top 5 Positivas (Suben el precio):")
    print(corr_pearson.head(6)[1:].to_string())
    print("\nTop 5 Negativas (Bajan el precio):")
    print(corr_pearson.tail(5).to_string())

    # 2. Multicolinealidad (Variables redundantes)
    print("\nAVISO: MULTICOLINEALIDAD (>0.8)")
    matriz_corr = df_num.drop(columns=['Precio']).corr().abs()
    # Desapilamos la matriz, ordenamos y quitamos los pares duplicados
    pares_altos = matriz_corr.unstack().sort_values(ascending=False).drop_duplicates()
    pares_peligrosos = pares_altos[(pares_altos > 0.8) & (pares_altos < 1.0)]
    
    if len(pares_peligrosos) > 0:
        print(pares_peligrosos.head(5).to_string())
    else:
        print("No se encontraron variables con correlación superior a 0.8.")

    # 3.1 Spearman (Si hay curvas monótonas claras)
    print("\n--- SPEARMAN (Relaciones No Lineales) ---")
    corr_spearman = df_num.corr(method='spearman')['Precio'].sort_values(ascending=False)
    print(corr_spearman.head(6)[1:].to_string())

    # 3.2 Kendall (Si son muestras pequeñas, empates o errores)
    print("\n--- KENDALL (Relaciones No Lineales) ---")
    corr_kendall = df_num.corr(method='kendall')['Precio'].sort_values(ascending=False)
    print(corr_kendall.head(6)[1:].to_string())

analizar_correlaciones_e_importancia(df_venta_limpio, "Venta")
analizar_correlaciones_e_importancia(df_alquiler_limpio, "Alquiler")

In [ ]:
for nombre, (df_orig, df_limp) in datasets_limpios.items():
    df_num_pred = df_limp.select_dtypes(include=["int64", "float64"]).drop(columns=["Precio"], errors="ignore")
    corr_matrix = df_num_pred.corr(method="spearman").abs()

    mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
    plt.figure(figsize=(18, 14))
    sns.heatmap(corr_matrix, mask=mask, cmap="Reds", vmin=0, vmax=1, annot=False, linewidths=0.3)
    plt.title(f"Multicolinealidad entre predictores (Spearman) — {nombre}")
    plt.tight_layout()
    plt.show()

    pares = corr_matrix.unstack()
    pares = pares[(pares > 0.85) & (pares < 1.0)].drop_duplicates().sort_values(ascending=False)
    print(f"\nPares con r > 0.85 — {nombre}:")
    print(pares.head(15).to_string())

Quitamos las columnas "dist_min_piscinas","cantidad_piscinas_cerca","estaciones_cerca" y "cantidad_alimentacion_cerca"

In [ ]:
COLS_MULTICOL = [
    "dist_min_piscinas",       # Mismo equipamiento deportivo
    "cantidad_piscinas_cerca", # Mismo equipamiento deportivo
    "estaciones_cerca",              # Líneas es más informativo
    "cantidad_alimentacion_cerca",   # Comercio es más general
]

df_venta_limpio    = df_venta_limpio.drop(columns=COLS_MULTICOL)
df_alquiler_limpio = df_alquiler_limpio.drop(columns=COLS_MULTICOL)

In [ ]:
print(df_venta_limpio.info())
print(df_alquiler_limpio.info())